# Chapter 60. Designing Whole Systems

*The runnable half of this chapter.* Every code cell below is the chapter's own code, and the words around it are the chapter's own explanation. Run the cells in order: each one uses names made by the cells before it.

Riverstone Supplies is the single worked example throughout the book, so the data here is the same data you meet in every other chapter.

Built from `manuscript/ch60-designing-whole-systems.md`.

# Chapter 60. Designing Whole Systems

*Part 7 — Architecture, Governance & Leadership*

> **Chapter at a glance**
>
> **You will learn to:** explain what "architecture" means once a decision affects more than one team's tools · draw and read a C4 diagram at the zoom level a conversation actually needs · turn a vague wish ("make it fast", "make it secure") into a number someone can test against, including a p95 and a count of allowed failures · write an architecture decision record (ADR) that a stranger could read in two years and understand why · design a whole system by composing the pieces earlier parts of this book built separately · evolve a design under real constraints — budget, headcount, a legacy system nobody is allowed to touch · recognize the failure patterns of over-engineering, resume-driven design, and the big-bang rewrite.
>
> **Before you start:** this chapter assembles systems you have already built. What it uses from each:
> - Chapter 16 (Power BI dashboards) and Chapter 20 (the Daily Sales Flash: sent by 07:30 IST on working days, and only after its four checks pass, section 20.11).
> - Chapter 21, section 21.3 (percentiles, and why service levels use them).
> - Chapters 45 to 47: the warehouse's raw, staging and mart layers; the Dagster pipeline that runs at 06:30 every morning (`riverstone_pipeline.py`); freshness judged against the previous working day (section 47.5).
> - Chapter 32, section 32.13 (the semantic layer: one tested definition of each metric).
> - Chapter 49 (the sensor archive as a Delta table, and the correction that failed without one) and Chapter 51 (reverse ETL into the CRM, `PATCH` versus `PUT`, systems of record).
> - Chapter 52 (AWS, region `ap-south-1`, the pipeline deployed as a scheduled container task).
> - Chapters 55 to 58: the support assistant and its refusal floor, the defect model's FastAPI service and shadow mode, and the PO-intake pipeline in assisted mode.
>
> You don't need to remember every detail: each piece is reintroduced in a line, with its chapter, where it's used.
>
> **Time needed:** 12–15 hours, spread over two weeks.
>
> **Tools:** nothing new to install. Two short calculations in section 60.3 use Python and pandas (Chapters 17 and 18), in a notebook in your usual virtual environment. Diagrams in this chapter can be drawn with a free tool (Mermaid, diagrams.net, or even a whiteboard); the companion files describe the same systems in plain text so you can redraw them yourself.
>
> **Practice data:** `companion/ch60/`: the full worked design document for a Riverstone Analytics & AI Platform, a blank ADR template, five filled-in architecture decision records, and a non-functional-requirements worksheet — all built from the real systems described in Parts 2, 5 and 6 of this book.

---

## Why this matters

Every chapter before this one taught you to build one good thing: a clean pipeline, a trained model, a dashboard that doesn't lie. Riverstone, over the course of this book, now has all of them — a warehouse, an orchestrator, a defect-detection model, a document assistant, an email-reading pipeline that writes into the ERP, a daily flash report. Each one, on its own, works.

Put them in the same company at the same time and a new kind of problem appears, one that no single chapter could have taught, because it isn't about any one piece. It's about the *seams*: what happens when the PO-intake pipeline's 06:00 run is still writing a heavy Monday's orders into the ERP at 06:31, and the orchestrator's 06:30 run is already reading the same order table? Who gets paged when the defect model's serving endpoint goes down — the person who built the model, or the person who owns the pipeline that feeds it? If the CRM sync and the support assistant both need the customer's current address, which one is allowed to be wrong for five minutes?

**Architecture is the discipline of answering those questions on purpose, before they happen, instead of by accident, at 2 a.m., in production.** It's not a fancier word for "system design" reserved for people with the word "Architect" in their title — it's what any senior analyst or engineer starts doing the moment their decisions stop affecting only their own script and start affecting everyone else's.

This chapter teaches the vocabulary and the habits: how to draw a system so someone else can understand it, how to turn "make it good" into something you can actually check, how to write down a decision so it doesn't get re-litigated every six months, and how to look at eight separately-built Riverstone systems and describe them as one platform.

---

## In plain English

Think about the difference between building one excellent room in a house and being the person who decides where the plumbing goes for the whole house.

A brilliant kitchen, on its own, doesn't need an architect. But the moment there's a kitchen, two bathrooms, and a laundry room, someone has to decide: which rooms share a wall so the pipes can be shared too? What happens if the water pressure drops when three taps run at once? If the kitchen is remodeled next year, which walls can move without the upstairs bathroom falling through the floor?

Nobody asks these questions while admiring one perfect room. They come from stepping back and looking at the whole house — and from having been the person who, at 2 a.m., got the call that the upstairs bathroom is now the downstairs kitchen's ceiling.

That step-back view — the whole house, not one room — is what this chapter, and this Part of the book, is about. The individual rooms (the pipelines, the models, the dashboards) were Parts 2 through 6. Part 7 is the plumbing diagram, the load-bearing walls, and the decision about which walls can move.

---

## 60.1 What "architecture" means, practically

**Architecture is any decision that's expensive to change later and affects more than one part of the system.** That's a working definition, not a formal one, and it's useful because it tells you exactly when you're doing architecture and when you're not.

Choosing a variable name is not architecture — it costs nothing to change and affects one function. Choosing Dagster as Riverstone's orchestrator (Chapter 46) *was* architecture — every pipeline, every team, and every future hire now has to know Dagster, and un-choosing it means rewriting all of them. Choosing to store the sensor archive as Delta Lake instead of plain Parquet files (Chapter 49) was architecture for the same reason: it's a decision that would be genuinely painful to reverse once fifty pipelines depend on it.

**Three properties usually mark a decision as architectural:**

1. **It's hard to reverse.** Not impossible — nothing is impossible — but expensive enough that "we'll just change it later" is not a credible plan.
2. **It constrains what comes after it.** Once the warehouse's tables are laid out in raw, staging and mart layers (Chapters 45 to 47), every later choice (the dbt models, the freshness checks, what the BI tool connects to) is built on that layout.
3. **Getting it wrong is expensive in a way that shows up late.** A bad architecture decision often looks fine for months, then fails exactly when the system is under the most load or the most scrutiny — the pattern behind nearly every "war story" in Chapters 45 through 58.

**What an architect actually does, day to day**, is less glamorous than the title suggests and more useful than most people expect:

- Asks "what happens when this fails?" before it's built, not after.
- Draws the system so the five people who each understand one piece of it can see how their pieces fit.
- Says no to a request that would work today and become unmaintainable in a year — and can explain why, in a way the requester accepts.
- Writes decisions down, so the next person doesn't have to reverse-engineer *why* from the code.

None of that requires a special title. Every analyst who's chosen a database schema, every engineer who's decided where a check belongs, has already been doing architecture in miniature. This Part scales the same judgment up to the level of an entire company's systems.

---

## 60.2 Reading and drawing C4 diagrams

The most common failure in system communication isn't a bad diagram — it's a diagram at the wrong **zoom level** for the conversation. Someone asks "how does data get from the plant to the dashboard?" and receives a diagram with forty labeled function calls, or asks "what does this one service do internally?" and receives a single box labeled "Backend."

The **C4 model** (Context, Containers, Components, Code), from software architect Simon Brown, fixes this by naming four fixed zoom levels and insisting you draw one at a time.

![Four boxes showing the C4 model's zoom levels: Context (the system as one box among the people and systems around it), Container (the system's major deployable pieces), Component (one container's internal modules), and Code (rarely drawn by hand)](figures/fig60-1-c4-levels.svg)

*Figure 60.1 — Zoom in one level at a time. Mixing levels — a context diagram with implementation detail crammed in — helps nobody.*

**Level 1: Context.** The system is one box. Everything else is either a person (a user) or another system it talks to. This is the diagram you show someone who needs to know *what the system is for*, not how it works.

![The Riverstone Analytics & AI Platform drawn as one box. Around it, people drawn as icons with their names underneath: branch managers, customers, sales and operations staff, and the data and analytics team. Other systems drawn as grey boxes: the ERP, the CRM, the plant sensors and Taloja line camera, and the email inbox. Every arrow runs between the platform and one of them: orders from the ERP and confirmed PO orders back to it, leads and segments from the CRM and lead scores and follow-up flags back to it, readings and images from the plants, PO and support emails from the inbox, dashboards and the Daily Sales Flash to branch managers, answers to customers, drafts to confirm and flags to sales and operations staff, and a dashed arrow from the data team, who build and run it.](figures/fig60-2-context-diagram.svg)

*Figure 60.2 — A context diagram for the platform this chapter designs. One box, and every arrow crosses its boundary. Nobody here asks which orchestrator runs inside — that question belongs one level down.*

At this level, a new employee, a manager, or an auditor can understand the whole picture in thirty seconds: the platform reads orders from the ERP, leads from the CRM, sensor readings and line-camera images from the plants, and emails from the inbox; it is read through dashboards and the Flash, and it writes back to the ERP and the CRM. Nothing about *how* appears yet, and that's the point.

**Level 2: Container.** Open the one box from level 1 and show its major deployable pieces — the things you could run, scale, or replace independently. "Container" here doesn't mean Docker specifically (though a container in this sense is often deployed as one, as Chapter 52 deployed the pipeline); it means a separately runnable unit.

![The platform opened into eight containers in two columns, each labelled AI or data: the PO-intake pipeline and the support assistant (AI); ingestion with Dagster and the warehouse of raw, staging and mart tables plus the Delta sensor archive (data); the defect model service (AI) and the semantic layer (data); BI and Flash, and the reverse-ETL syncs (data). Grey boxes at the edges are outside systems: the email inbox, the ERP, the plant sensors and Taloja line camera, the policy and product spec documents, and the CRM. Numbered arrows trace one purchase order: 1 email inbox to PO intake; 2 a person confirms the draft; 3 PO intake to the ERP; 4 ERP to ingestion and ingestion to the warehouse; 5 warehouse to semantic layer; 6 semantic layer to BI and Flash and to the reverse-ETL syncs, and from them to the CRM. Dashed grey arrows show other flows: sensor files to ingestion, camera images to the defect model, documents to the support assistant. A band underneath lists shared services used by every container: credential vault, run logs, alerting, monitoring.](figures/fig60-3-container-diagram.svg)

*Figure 60.3 — Opening the context box. This is the level where "what talks to what, and how" gets decided — and where most of this book's later chapters live. The numbered arrows are the purchase-order trace in section 60.5.*

The eight containers, each in one line:

- **Ingestion (Dagster, Chapter 46):** the 06:30 run that copies ERP orders, CRM leads, and the dispatch and sensor files into the warehouse.
- **Warehouse (Chapters 45, 47 and 49):** the raw, staging and mart tables, plus the sensor archive, a Delta table in object storage.
- **Semantic layer (Chapter 32):** the dbt models that hold one tested definition of each metric, such as revenue and active customer.
- **BI and Flash (Chapters 16 and 20):** the Power BI dashboards and the Daily Sales Flash email.
- **Defect model service (Chapters 53 and 56):** scores each part's image from the Taloja line camera, behind a small web service built with **FastAPI**, the Python library Chapter 56 used to answer requests over HTTP.
- **Support assistant (Chapter 55):** answers questions using **RAG** (retrieval-augmented generation): it first finds the relevant passages in Riverstone's policy and spec documents, then answers only from them.
- **PO-intake pipeline (Chapter 58):** drafts orders from customers' purchase-order emails; a person confirms each one before it is written to the ERP.
- **Reverse-ETL syncs (Chapter 51):** write lead scores, follow-up flags and customer segments from the warehouse back into the CRM.

Notice what the container diagram makes visible that the context diagram hid: there are **three AI containers** (the defect model, the support assistant, and the PO-intake pipeline) beside **five data containers** (ingestion, warehouse, semantic layer, BI and Flash, and the reverse-ETL syncs), built in Parts 2, 5 and 6. A reader who only saw the context diagram would have no idea the system had grown this complex inside; a reader who only saw a component-level diagram of one container would have no idea it was one piece of eight.

**Level 3: Component.** Open one container and show the modules inside it and how they call each other — for example, the PO-intake pipeline's read/extract/parse/validate/decide/write/report stages from Chapter 58 (section 58.3). You draw this level for the team that owns that one container, not for the whole company.

**Level 4: Code.** The actual classes or functions. The C4 method's own advice, and this book's: **don't draw this by hand.** Generate it from the code if a tool can, or skip it — a hand-maintained **UML class diagram** (a standard drawing of a program's classes and how they relate) goes stale within a month and actively misleads after that.

**The discipline that matters more than the notation:** before drawing anything, ask *who is this for, and what do they need to decide?* A board member needs level 1. A new hire on the data platform team needs level 2, and level 3 for whichever container they'll work on first. Nobody, ever, needs level 4 drawn by hand.

---

## 60.3 Non-functional requirements: numbers, not adjectives

A **functional requirement** says what the system does: "send the daily flash email," "score an image for defects." A **non-functional requirement (NFR)** says how well it has to do it: how fast, how available, how secure, how much it may cost. Most systems fail not because the functional requirements were wrong, but because the non-functional ones were never written down, so nobody could tell whether the system met them.

The problem is that non-functional requirements almost always arrive as adjectives — "fast," "reliable," "secure," "scalable" — and adjectives can't be tested. **The job of this section is turning each adjective into a number.**

![Five vague requirements turned into precise, testable ones: "make it fast" becomes the Daily Sales Flash sent by 07:30 IST on working days, p95 over the month; "make it reliable" becomes at most 2 pipeline runs a year needing a person to step in, counted monthly from Dagster's run history; "make it secure" becomes no analyst's login reading another branch's customer data, reviewed quarterly; "make it scale" becomes the 06:30 run finishing by 07:15 and the dashboard's first view loading in under 3 seconds, p95, at 5 times today's order volume; "keep it cheap" becomes platform cost under 0.50 rupees per 1,000 order lines, reviewed monthly](figures/fig60-4-nfr-precision.svg)

*Figure 60.4 — Every one of these vague requirements was actually said, in some form, about a real Riverstone system in Parts 2, 5 or 6. The right-hand column is what an architect writes down instead.*

### Reading a p95

Two of those requirements end in **p95**, short for the **95th percentile**: sort all the measurements, and p95 is the value that 95% of them are at or below. Chapter 21 (section 21.3) showed why service levels are written this way: they describe what people actually experience, not the average. Chapter 56 used the same number for the defect model's response time.

Work one through for the Flash. Suppose that over one month of 20 working days, the Flash went out these many minutes after 07:00. The deadline, 07:30, is 30 minutes. By hand first: sorted, the 19th value is 18 and the 20th is 42 (the morning the source load ran late). Chapter 21's position rule puts the 95th percentile at position 1 + 0.95 × (20 − 1) = 19.05, a twentieth of the way from 18 to 42: 18 + 0.05 × 24 = 19.2 minutes. Now the same in pandas. Before you run it, predict: will the average be above or below the p95?

In [1]:
import pandas as pd

sent = pd.Series([12, 9, 15, 11, 10, 14, 13, 9, 11, 42,
                  12, 10, 16, 11, 13, 12, 9, 18, 10, 14])
print("sorted:", sorted(sent))
print("average:", sent.mean())
print("p95:", round(sent.quantile(0.95), 1))

sorted: [9, 9, 9, 10, 10, 10, 11, 11, 11, 12, 12, 12, 13, 13, 14, 14, 15, 16, 18, 42]
average: 13.55
p95: 19.2


How it works:

- `pd.Series([...])` holds the 20 send times, one per working day, as in Chapter 18. The numbers are made up for this example.
- `sorted(sent)` returns the values in increasing order, so you can check the 19th and 20th against the hand calculation.
- `sent.mean()` is the average.
- `sent.quantile(0.95)` is the 95th percentile, with the same position rule as Chapter 21; `round(..., 1)` keeps one decimal place.

**Reading it.** p95 is 19.2 minutes, the same as the hand calculation, so the month meets "sent by 07:30, p95". One freak morning (42 minutes, a Flash at 07:42) did not fail the month, which is what you want from a target: it should catch a pattern, not punish one bad day.

Now the other direction. What if you change two ordinary mornings into late ones, 35 and 38 minutes? Predict both numbers, then run it:

In [2]:
late_month = sent.copy()
late_month[3] = 35
late_month[7] = 38
print("average:", late_month.mean())
print("p95:", round(late_month.quantile(0.95), 1))

average: 16.2
p95: 38.2


- `sent.copy()` makes a separate Series, so changing it leaves `sent` as it was.
- `late_month[3] = 35` replaces the value with index 3 (the fourth day, 11 minutes) with 35; `late_month[7] = 38` does the same for the eighth day.

**Reading it.** The average hardly moved: 16.2 minutes still looks comfortable. But three late mornings in a month is a pattern, and p95 shows it at once: 38.2 minutes, a Flash at 07:38, and the month fails. **That is why a target names p95 and not the average: bad days can't hide behind good ones.**

### A template for any adjective

| Adjective | Ask | Turns into |
|---|---|---|
| Fast | How fast, measured how, for what share of requests? | "p95 latency under 200 ms" |
| Reliable | How many failures are acceptable, counted over what window? | "At most 2 scheduled runs a year need a person to step in, counted monthly" |
| Secure | Secure against what, specifically? | "No credential can read data outside its own branch" |
| Scalable | To what future volume, by when? | "At 5 times today's order volume, the 06:30 run still finishes by 07:15 and the dashboard's first view loads in under 3 s, p95" |
| Cheap | Compared to what, and is this a ceiling or a target? | "Under ₹0.50 per 1,000 order lines processed" |

**Why "reliable" became a count.** "99.5% of scheduled runs succeed, measured monthly" sounds precise, but work it out for Riverstone. The pipeline is scheduled once a day (Chapter 46), about 30 runs a month, so one failed run is 1 ÷ 30 = 3.3% of the month: a 96.7% success rate. A monthly 99.5% target therefore allows no failures at all, which nobody means. Over a year of 365 runs, 99.5% allows 365 × 0.005 = 1.8 failures. When there are few events, **write the target as a count**: at most 2 runs a year need a person to step in.

Two categories are worth calling out because they're the ones systems most often get wrong silently.

**Availability** is the share of time a system is up, usually expressed in "nines". By hand first, for 99.9%: a year has 365 × 24 × 60 = 525,600 minutes; 0.1% of that is 525.6 minutes, about 8.76 hours of downtime a year. The cell below does the same for four levels, per year, per month and per week. Before you run it, predict: roughly how long a month can a system at 99.5% be down?

In [3]:
minutes_per_year = 365 * 24 * 60

def readable(minutes):
    return f"{minutes / 60:.2f} hours" if minutes >= 60 else f"{minutes:.1f} minutes"

print(f"minutes in a year: {minutes_per_year:,}")
for availability in [99, 99.5, 99.9, 99.99]:
    down = minutes_per_year * (100 - availability) / 100
    print(f"{availability}%: {readable(down)} a year, {readable(down / 12)} a month, "
          f"{readable(down / 52)} a week")

minutes in a year: 525,600
99%: 87.60 hours a year, 7.30 hours a month, 1.68 hours a week
99.5%: 43.80 hours a year, 3.65 hours a month, 50.5 minutes a week
99.9%: 8.76 hours a year, 43.8 minutes a month, 10.1 minutes a week
99.99%: 52.6 minutes a year, 4.4 minutes a month, 1.0 minutes a week


How it works:

- `minutes_per_year` is the hand calculation above.
- `readable()` shows a number of minutes in hours when it is an hour or more, and in minutes otherwise. `... if minutes >= 60 else ...` is the conditional expression from Chapter 17.
- `100 - availability` is the share of time the system may be down, in percent, so `down` is the minutes of downtime allowed in a year. `down / 12` is a month and `down / 52` a week.
- `{minutes_per_year:,}` prints the number with thousands separators, and `:.2f` and `:.1f` round to two and one decimal places.

**Reading it.** Each additional nine cuts the allowed downtime by ten, and costs disproportionately more to achieve: a spare server, automatic failover, someone always reachable to fix it. That is exactly why an architect's job includes asking *does this system actually need 99.99%*, not assuming more is always better. Riverstone's daily flash email genuinely doesn't need four nines; a plant safety alert genuinely might. To see another level, add it to the list (try 99.95) and run the cell again.

**Freshness** — how stale can this number be before it's wrong? — is a non-functional requirement people forget to state and then argue about after the fact. Chapter 47's freshness check (section 47.5) exists precisely because "the data should be current" had never been turned into a promise: at the 06:30 run, the newest orders must be from the previous working day, so the Flash can go out by 07:30. (Chapter 61 uses the word *consistency* for a narrower, technical idea, so this chapter keeps to *freshness*.)

> **Watch out: an NFR with no owner and no test is a wish, not a requirement.** Writing "a 99.5% run-success target" in a document and never measuring it against reality is worse than not writing it at all, because it creates false confidence. Every NFR in this chapter's worked example (section 60.5) has a stated measurement method.

---

## 60.4 Trade-offs and the architecture decision record

Almost no architectural choice is strictly better than its alternative — if it were, there'd be nothing to decide. Every real choice **trades one good property for another**: Delta Lake trades operational simplicity (one more tool to run) for reliability (atomic writes, and **time travel**, reading the table as it was at an earlier version, Chapter 49) that plain Parquet files don't have. Dagster trades a learning curve for testability and observability that a folder of cron jobs doesn't have. There is no version of this book, or this platform, where every choice is free.

**An architecture decision record (ADR)** is a short, permanent note that captures one such choice: what was decided, why, what else was considered, and what the consequences are. Its value isn't in the decision itself — it's in **making the reasoning outlive the meeting where it was made.**

![A filled-in ADR, ADR-014, choosing a Delta Lake table for the sensor archive. Its rows: status and date (accepted the week after the M-07 correction failed); context (216,000 readings a day from 25 machines; a job rewriting six weeks of M-07's readings failed partway under plain Parquet, leaving 11 days fixed, 8 missing and 23 uncorrected, and recovery took two days); decision; three alternatives considered and why each was rejected (plain Parquet with folder versions, Apache Iceberg, an ordinary warehouse table); consequences; and owner and revisit trigger](figures/fig60-5-adr-example.svg)

*Figure 60.5 — One real decision from Part 5, written up properly. Two years from now, nobody has to guess why the sensor archive uses a format the rest of the warehouse doesn't.*

**The seven parts of an ADR** (adapted from Michael Nygard's format, which has a title, status, context, decision and consequences; this book adds alternatives considered, and owner and revisit trigger):

1. **Title and number** (ADR-014), so it can be referenced from code, tickets, or other ADRs.
2. **Status** (proposed, accepted, superseded) and the date.
3. **Context** — the situation that made a decision necessary. Chapter 49's context wasn't hypothetical: a real correction run failed mid-way under plain files and cost two days of recovery.
4. **Decision** — one sentence, stated plainly.
5. **Alternatives considered**, each with why it was rejected. This is the part people skip, and it's the part that prevents the same debate from happening again in eight months when someone new joins and asks "why don't we just use Iceberg?"
6. **Consequences** — including the costs of the decision, not only its benefits. Every decision has a cost; naming it is what makes the ADR honest rather than promotional.
7. **Owner and revisit trigger** — who answers for the decision, and the specific future condition that would make it worth reconsidering.

**When to write one:** any time a decision meets section 60.1's test — hard to reverse, affects more than one part of the system, or expensive to get wrong. Not every choice needs one; a folder of a hundred ADRs for trivial decisions is as useless as none at all. Riverstone's data platform team, by the time this design document was written, had accumulated twenty-six — one for the orchestrator, one for the table format, one for the cloud region, and so on, each traceable to a specific chapter's story.

---

## 60.5 The worked example: designing the Riverstone Analytics & AI Platform

Everything so far has been vocabulary. This section uses it on a real design problem. By the time Part 6's projects are running, Riverstone has built a warehouse and orchestrator (Part 5), a defect-detection model, a document assistant, and an email-reading order pipeline (Part 6), and a set of dashboards and automations (Part 2) — separately, by different people, at different times, for different immediate needs. Nobody has ever drawn what all of it looks like *together*, and two incidents in one month (the PO-intake run still writing orders while the 06:30 pipeline run read them, and a support-assistant answer that quoted a superseded delivery policy, Chapter 55's story) made clear that "together" is now a real system, whether anyone designed it as one or not.

**The design document below is what an architect produces in that situation** — not to invent new systems, but to describe, constrain, and connect the ones that already exist. The full version, with every section expanded, is `companion/ch60/design-document.md`; what follows is the document's spine.

### Purpose and scope

*One paragraph, always first, always readable by someone with no context.* "This document describes the Riverstone Analytics & AI Platform: the combined set of data pipelines, models, and delivery systems that turn order, sensor, and document data into reports, alerts, and automated actions across sales, operations, and support. It does not cover the ERP or CRM themselves, which remain systems of record owned by their respective vendors and teams." A **system of record** (Chapter 51, section 51.5) is the one system allowed to be the source of truth for a piece of data.

### Requirements

**Functional**, in one line each: ingest orders, sensor readings, and CRM data reliably; detect line-level product defects from images; answer support and policy questions from current documents only; read purchase-order emails and write confirmed orders to the ERP; send the Daily Sales Flash; keep the CRM's lead scores current.

**Non-functional**, using section 60.3's method:

| Requirement | Precise form | Measured how |
|---|---|---|
| Freshness | At the 06:30 run, the newest orders are from the previous working day | Freshness check on every run, before the Flash is built (Ch 47, section 47.5) |
| Pipeline reliability | At most 2 runs a year need a person to step in (99.5% of 365 daily runs would allow 1.8) | Counted monthly from Dagster's run history (Ch 46) |
| Flash delivery | Sent by 07:30 IST on working days, p95 over the month, and only after its four checks pass | From the send time in the run log (Ch 20, Ch 46) |
| Defect model latency | p95 under 50 ms per image, at line speed: a budget with room to spare, since Chapter 56's service answered in a few milliseconds | Load test before each deployment (Ch 56) |
| Assistant grounding | Refuses rather than guesses when no document's score clears the absolute floor | Golden-set evaluation, every deploy (Ch 55, Ch 57) |
| PO-intake accuracy | No order is written to the ERP without a named person confirming it (assisted mode); orders over ₹1,00,000 also need approval | Daily count of ERP writes with no confirmer = 0; a weekly sample of 20 confirmed orders checked against their emails (Ch 58) |
| Cost ceiling | Platform cost under ₹0.50 per 1,000 order lines processed | Monthly cost review (Ch 65) |
| Access control | No credential can read another branch's customer data | Quarterly access review (Ch 64) |

### The architecture: context and containers

Figure 60.2 (context) and Figure 60.3 (container) *are* this section — a design document's diagrams are not illustrations of the text, they're load-bearing parts of it. The prose around them explains what isn't obvious from the boxes:

- **Ingestion, warehouse, and semantic layer are shared infrastructure** that every other container reads from or writes to. They are the one part of the platform that, if it goes down, everything else eventually stops being trustworthy.
- **The three AI containers are independent of each other.** The defect model doesn't know the support assistant exists, and neither depends on the PO-intake pipeline. This is deliberate: a failure in one shouldn't cascade into the others, and each was built and can be redeployed on its own schedule.
- **Two containers write to systems outside the platform's control:** the PO-intake pipeline writes orders to the ERP, and the reverse-ETL syncs write scores, flags and segments to the CRM. Both are held to a stricter standard than read-only containers, because a wrong write has consequences a wrong read doesn't. Three rules, from Chapters 51 and 58: an **idempotency key** on every write, a value built from the source (for PO intake, the source email) so that writing the same thing twice creates only one record; a named system of record for every field; and **reconciliation**, checking after each run that what was written matches what was meant.

### Key decisions (the ADR index)

A real design document doesn't re-argue each decision inline; it links to the ADR and states the conclusion. The platform's index, abbreviated:

| ADR | Decision | Chapter |
|---|---|---:|
| ADR-002 | Orchestrator: Dagster | 46 |
| ADR-007 | Cloud region: AWS `ap-south-1` (Mumbai) | 52 |
| ADR-014 | Sensor archive table format: Delta Lake | 49 |
| ADR-018 | Data quality: SQL tests inside the pipeline (write–audit–publish), not a dedicated observability platform (yet) | 47 |
| ADR-021 | PO-intake mode: assisted, not straight-through, given a 19% silent-error rate at full automation | 58 |
| ADR-023 | Support assistant: refuse on absolute score floors, not normalized scores; mark superseded documents at index time | 55 |
| ADR-026 | CRM writes: `PATCH` only, one system of record per field, after a sync's `PUT` erased a negotiated discount | 51 |

Reading down this table is, itself, a compressed history of Parts 5 and 6: almost every decision traces to an incident, a measurement, or a finding that a simpler approach didn't hold up. **That's what a healthy ADR index looks like** — not a list of things that sounded good, but a list of things that were tested against reality and survived.

### Data flow: one request, traced end to end

Design documents earn their keep when they can answer "what actually happens when—?" Trace one: *a customer emails a purchase order.* The step numbers match the numbered arrows in Figure 60.3.

1. The PO-intake pipeline's 06:00 weekday run reads the new emails from the mailbox (in Chapter 58, a folder stood in for it). It checks on a timer rather than being told when an email arrives, which is called **polling** (Chapter 51, section 51.6).
2. Extraction and validation run. The email becomes a draft order for the coordinator to confirm (Chapter 58's assisted mode), or goes to the exception queue with a reason.
3. On confirmation, the order is written to the ERP, with the source email as its idempotency key (Chapter 58, section 58.4), so a re-run or a retry can never create a duplicate.
4. The next 06:30 ingestion run (Chapter 46) copies the new order from the ERP into the warehouse.
5. The semantic layer's models recompute the affected totals.
6. The daily Flash, the dashboards, and the reverse-ETL syncs to the CRM all read from the same, now-updated, semantic layer — so the sales rep's CRM record and the branch manager's dashboard agree, because they share one upstream source rather than being computed twice.

Nowhere in that chain does one container reach into another's internals. Every numbered arrow in the container diagram is a real, traceable step in this walkthrough — which is exactly the test a container diagram should pass: **if you can't trace a real request across it, the diagram is decoration, not documentation.**

### Risks and open questions

A design document that only lists what's decided is marketing. It should also say what isn't:

- **The PO-intake exception queue has no defined maximum wait time.** If email volume triples during a festive season, does the human-confirmation step become the bottleneck? Not yet load-tested.
- **The defect model and the support assistant share no infrastructure today**, which is good for isolation and bad for cost — two separate serving setups, each built and looked after on its own. Worth revisiting once a third model exists.
- **Chapter 64 (this Part) has not yet defined the platform's access-control model**, so the "no credential can read another branch's data" requirement above is aspirational until that chapter's work lands.
- **The cost ceiling was set top-down and has never been priced bottom-up.** Nobody has yet added up what the platform's servers, storage and services actually cost per order line. Chapter 65 does this.

---

## 60.6 Evolving a design under real constraints

The design in section 60.5 describes what Riverstone's platform *should* look like. Getting there from eight separately-built systems, on a real budget, with a small team, is a different problem — and it's the one architects spend most of their time on, because **greenfield** designs (a brand-new system, with nothing already running) are rare and everything else is evolution.

**Three constraints shape almost every real redesign:**

- **Budget.** The "right" answer (a dedicated feature store, a managed observability platform, a second cloud region for redundancy) often costs more than the problem it solves is worth. Part of the job is sizing the fix to the pain, not to the textbook.
- **Headcount.** A four-person data team cannot operate the same number of moving parts as a forty-person one, no matter how good the architecture diagram looks. Fewer, more standardized containers usually beat more, more specialized ones, for a small team.
- **Legacy systems nobody can touch.** The ERP is a vendor system; Riverstone can integrate with it but not redesign it. Good architecture works *around* that constraint openly (as the PO-intake pipeline does, treating the ERP as a system of record it writes to carefully) rather than pretending the constraint doesn't exist.

**A pattern for evolving safely, used throughout Parts 5 and 6 without always naming it:**

1. **Run the old and new side by side** before switching. Chapter 56's **shadow mode** (section 56.6) is this pattern applied to a model: the new version scores real traffic beside the old one, but only the old one's answer is used. The same idea applies to a new pipeline or a new table format.
2. **Migrate the lowest-risk piece first**, to learn the real cost of the change before betting something important on it.
3. **Keep a rollback path** until the new piece has run through at least one full cycle of whatever could go wrong (a month-end close, a festive season, a provider outage).
4. **Write the ADR before, not after** — deciding under pressure, mid-migration, is how the alternatives-considered section gets forgotten.

**Resisting the urge to redesign everything at once** is itself a skill. The temptation, once you can see the whole system clearly for the first time, is to fix all of it — new orchestrator, new warehouse, new everything, all at once, on the strength of one good diagram. That is the **big-bang rewrite**: replacing a whole working system in one go instead of piece by piece, as if Riverstone froze its pipeline, dashboards and PO intake for two quarters while their replacements were built. It almost never survives contact with a real company's calendar. Its cousin is **resume-driven design**: adopting a tool because it's exciting to learn, not because the current one failed — a Kafka cluster (Chapter 50) for a report that runs once a day. Both are in this chapter's mistakes table below.

---

## Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Drawing the wrong zoom level | A board member glazes over at a component diagram; an engineer can't build from a context diagram | Ask who it's for and what they need to decide, before drawing anything |
| Treating every decision as architectural | A hundred ADRs, most for trivial choices, that nobody reads | Reserve ADRs for decisions that are hard to reverse or cross team boundaries |
| Non-functional requirements left as adjectives | "Make it fast" with no way to know if it succeeded | Turn every adjective into a number and a measurement method |
| An NFR with no owner | A requirement nobody checks, quietly false for months | Name who measures it and how often |
| Skipping "alternatives considered" in an ADR | The same debate happens again in eight months | Always record what was rejected, and why |
| A container diagram nobody can trace a real request through | The diagram is decorative, not documentation | Walk one real request across it before calling it done |
| Over-engineering for a scale that doesn't exist yet | Months spent on redundancy a four-person team can't operate | Size the design to the current constraint, with a documented path to grow |
| Resume-driven design | A new tool adopted because it's exciting, not because the old one failed | Require a real, named problem the new tool solves that the old one doesn't |
| The big-bang rewrite | Months of work, nothing shippable, and the old system rots waiting | Migrate the lowest-risk piece first; run old and new side by side |
| Ignoring legacy constraints | A design that assumes the ERP can be redesigned, when it can't | Design around real constraints, stated openly, not around a wish list |
| No rollback path during migration | A change that can't be undone if it goes wrong mid-flight | Keep the old path live until the new one has survived a full cycle |
| Deciding under pressure, mid-incident | The ADR gets written after the fact, missing the real alternatives | Write the ADR before the change, even a short one, not after |

---

## In the real world: the diagram that almost caused a rewrite

By the January after the PO-intake pilot went live, Meera Iyer had spent her whole time at Riverstone, since joining as a sales coordinator, building pieces of what was now, without anyone having planned it, a genuine platform: the warehouse and Dagster pipelines from her data engineering work, the defect model and support assistant from the AI projects, the PO-intake pipeline that had quietly become the busiest thing she'd ever built. Arvind Kapoor, the managing director, watching the company's data work outgrow "the analytics team's scripts," created a new role — Head of Data Platform — and, on Anita Rao's recommendation, asked Meera to take it and to figure out what came next.

Meera's first act in the new role was the one this chapter recommends: she drew the container diagram. Not because a diagram would fix anything by itself, but because nobody — including her — had ever seen the whole thing laid out at once.

The diagram was, by her own later admission, "genuinely alarming." Eight containers, built at eight different times, by processes that had made sense individually and looked chaotic together. Two jobs wrote to the CRM — the lead-score sync and Chapter 51's customer-segment sync — with different retry logic. The defect model and the support assistant each ran their own bespoke serving setup, duplicating infrastructure neither team had realized the other had built. And the rule about which system owned which CRM field, written after Chapter 51's erased-discount incident, lived only in a small internal document that the newer jobs had never been checked against.

Her first instinct, looking at that diagram, was to propose exactly what section 60.6 warns against: a single unified platform, rebuilt properly from the ground up, over the next two quarters. She got as far as drafting the proposal before a conversation with Vikram Singh stopped her. His question was simple: *"What happens to the daily flash, the defect alerts, and the PO intake while you're rebuilding all of it?"*

She didn't have a good answer, because there wasn't one. A two-quarter rewrite meant two quarters of a frozen platform, or two parallel platforms to maintain, on a four-person team, while the business kept running on the messy-but-working version underneath.

What she did instead, over the following two months:

1. **She wrote ADR-026 first.** It recorded the rule Chapter 51's incident had produced (`PATCH` only, one system of record per field), which until then lived only in that small internal document, and extended it to every container that writes to the CRM, with one retry policy for both syncs. It took a week and touched nothing else.
2. **She consolidated the two model-serving setups** into one shared FastAPI service, migrating the lower-risk one first: the defect model, because if the new service failed, the QC inspector could check parts by hand for a shift, while a wrong answer from the support assistant would go straight to a customer. She ran it in shadow mode for two weeks before cutting over — exactly Chapter 56's pattern, applied to infrastructure rather than a model.
3. **She left the eight-container shape alone.** It wasn't elegant, but every container had a real owner and a real reason to exist, and nothing about the diagram's *shape* was actually causing the incidents. The incidents came from three specific, nameable gaps — the CRM write rule that lived only in a side document, the duplicated serving infrastructure, and the undocumented retry difference between the two CRM syncs — not from the system's overall design.

The platform a year later looked almost the same as the "alarming" diagram from January, with three targeted fixes and an ADR behind every non-obvious choice: twenty-six of them by the time the design document in section 60.5 was finished. Nobody remembers the rewrite proposal, because it never happened, and nothing was lost by not doing it.

What made the difference:

- **She drew the whole picture before deciding anything**, which turned a vague unease into three specific, fixable problems.
- **She asked what a full rewrite would cost while it was happening**, not only what it would achieve when finished.
- **She fixed what was actually broken**, not what merely looked untidy on a diagram — an untidy diagram of a working system is not the same problem as a broken one.
- **She wrote the decisions down**, so the next person to look at that container diagram would understand why it looks the way it does.

---

## Project: design document for Riverstone's platform

**Goal:** produce a real design document — context diagram, container diagram, non-functional requirements, an ADR index, and a traced data flow — for a system you actually work with.

### Tools you'll need

- **Diagramming:** Mermaid (text-based, renders in many tools including GitHub), diagrams.net (formerly draw.io, free and works offline), or a whiteboard photographed and cleaned up. No paid tool is required for anything in this chapter.
- **ADR templates:** this chapter's seven-part format, adapted from Michael Nygard's short Markdown template; `adr-tools` is a free command-line helper for numbering and indexing ADRs, if you want automation.
- **Companion files (`companion/ch60/`):**
  - `design-document.md`: the full worked Riverstone Analytics & AI Platform design document, expanded from section 60.5.
  - `adr-template.md`: a blank, seven-part ADR template.
  - `adr-examples/`: five filled-in ADRs from the platform's index (orchestrator, cloud region, PO-intake mode, assistant refusal, CRM write rule); the table-format ADR is Figure 60.5.
  - `nfr-worksheet.md`: the adjective-to-number worksheet from section 60.3, blank and ready to fill in for your own system, with the availability table.

**Option A: your own system.** Pick something at your workplace that has grown past one person's or one team's full understanding — exactly the situation this chapter's worked example describes.

**Option B: Riverstone.** Extend section 60.5's document: draw the component-level diagram (level 3) for the PO-intake pipeline, using Chapter 58's read/extract/parse/validate/decide/write/report stages as your components.

**Steps**

1. **Write the purpose and scope paragraph first.** If you can't write it in three sentences a stranger would understand, the system isn't scoped clearly enough yet to design.
2. **List the functional requirements**, one line each.
3. **Turn at least five non-functional requirements from adjectives into numbers**, using section 60.3's table as a template, each with a stated measurement method and owner. Where events are few (a daily job), write the target as a count.
4. **Draw the context diagram** (level 1): one box, every person and system around it.
5. **Draw the container diagram** (level 2): the deployable pieces, and how they connect.
6. **Trace one real request end to end** across your container diagram, the way section 60.5 traced a purchase-order email, and number the arrows it uses. If you can't trace it, the diagram needs work.
7. **Write at least three ADRs** for real decisions already made (even informally) in this system, including the alternatives that were considered and rejected.
8. **List at least two open risks** honestly — something not yet solved, not yet load-tested, or not yet decided.

**What good looks like:** a reader with no prior context can understand what the system does from the purpose paragraph alone; every box on the container diagram has a one-line reason to exist; every NFR has a number and a way to check it; every ADR explains what was rejected, not only what was chosen; the risks section says something true and slightly uncomfortable, not "no known risks."

**Stretch goals**

- Present your design document to a colleague who's never seen it, and time how long it takes them to answer "what happens if this one container goes down?" A well-designed document makes that answer fast.
- Find a decision in your system that was never written down, interview whoever made it (or reconstruct the reasoning if they've left), and write the ADR retroactively.
- Redraw your container diagram as it would look in two years if current growth continues, and mark which containers would need to change first.

---

## Recap

- **Architecture is any decision that's hard to reverse and affects more than one part of a system** — not a title, a habit available to anyone whose choices reach beyond their own code.
- **The C4 model** gives four fixed zoom levels — context, container, component, code — so a diagram matches what its reader actually needs to decide.
- **Non-functional requirements** must become numbers ("sent by 07:30, p95 over the month", "at most 2 failed runs a year"), or they're wishes, not requirements — and every NFR needs a stated owner and measurement method. **p95** is the value 95% of measurements are at or below, so bad days can't hide behind an average; when events are few, a count is clearer than a percentage.
- **An ADR** captures a decision's context, the decision itself, the alternatives considered and rejected, its consequences, and who owns it — so the reasoning outlives the meeting where it happened.
- **The worked Riverstone platform** ties together Part 5's data infrastructure and Part 6's AI systems into one described whole, with a traceable data flow and an honest risks section.
- **Evolving a real system** means working within budget, headcount, and legacy constraints — migrating the lowest-risk piece first, running old and new side by side, and keeping a rollback path.
- **The temptation to rewrite everything** after finally seeing the whole picture is common and usually wrong: fix what's actually broken, not what merely looks untidy on a new diagram.

---

## Key terms

architecture · non-functional requirement (NFR) · functional requirement · C4 model · context diagram · container diagram · component diagram · code diagram · zoom level · UML class diagram · p95 (95th percentile) · availability ("nines") · freshness · architecture decision record (ADR) · trade-off · system of record · idempotency key · reconciliation · polling · greenfield · resume-driven design · big-bang rewrite · shadow mode (Chapter 56) · rollback path

*(All terms are defined in the Glossary, Appendix A.)*

---

## Check yourself

- [ ] You can say, for any decision, whether it's architectural using the "hard to reverse, crosses boundaries, expensive to get wrong late" test.
- [ ] You draw C4 diagrams at the zoom level the audience needs, and never mix levels on one diagram.
- [ ] You can turn "make it fast/reliable/secure/cheap" into a number and a measurement method, for any system you're asked about.
- [ ] You can explain what a p95 is, compute one, and say why a target names it instead of the average.
- [ ] You write ADRs that include what was rejected and why, not only what was chosen.
- [ ] You can trace a real request across a container diagram and know it's accurate because you did.
- [ ] You evaluate a redesign proposal by asking what it costs while it's happening, not only what it achieves when finished.
- [ ] You can name the difference between fixing what's broken and rebuilding what merely looks untidy.
- [ ] You resist over-engineering for a scale that doesn't exist yet, and resume-driven adoption of tools that don't solve a named problem.

---

## Exercises

### Warm-up

1. Using the three-property test in section 60.1, decide whether each is architectural: (a) renaming a Python variable; (b) choosing which cloud region to deploy in; (c) changing a chart's color; (d) choosing the message format two services use to talk to each other.
2. What's the difference between a level 1 (context) and level 2 (container) C4 diagram? Give an example of a question each one answers that the other doesn't.
3. Turn these into testable non-functional requirements: (a) "the dashboard should load quickly"; (b) "the system should be secure"; (c) "we shouldn't lose data."
4. List the seven parts of an ADR from memory, in order.
5. Why does an ADR need an "alternatives considered" section? What goes wrong without one?

### Core

6. Draw a context diagram (level 1) for a system you use or have built, showing at least three external people or systems.
7. Open one box from your context diagram into a container diagram. What are the deployable pieces, and how do they connect?
8. Write three non-functional requirements for that same system, each with a specific number and a stated way to measure it.
9. Trace one real request across your container diagram from end to end, the way section 60.5 traced a purchase-order email. Where, if anywhere, does the trace get vague or uncertain?
10. Write a full ADR for a real decision in a system you know, including at least two rejected alternatives.
11. Add 99.95 to the list in section 60.3's availability cell and run it. How much downtime a month does 99.95% allow, and how does that compare with 99.5%? For a system you know, which level is actually appropriate, and why?
12. Riverstone's platform has three AI containers that don't depend on each other (section 60.5). What's the architectural benefit of that independence, and what's given up by not sharing infrastructure between them?
13. A colleague proposes replacing Riverstone's orchestrator because a newer tool "looks better in the documentation." Using section 60.6's evolution pattern, what would you ask before agreeing?

### Stretch

14. Write the component-level (level 3) diagram for the PO-intake pipeline, using Chapter 58's stages. What would a level 4 (code) diagram of the "validate" stage need to show, and why does this chapter recommend not drawing it by hand?
15. Design a migration plan for consolidating two duplicated pieces of infrastructure (like Meera's two model-serving setups in the story), following section 60.6's four-step pattern.
16. Riverstone's design document (section 60.5) lists four open risks. Propose how you'd investigate and close one of them, and what evidence would let you remove it from the risks list.

### Think about it

17. A stakeholder asks you to skip writing the ADR because "we all agree, let's just build it." What do you say, and what would you be willing to compromise on (format, length) to get the decision recorded anyway?
18. Is a big-bang rewrite ever the right call? Under what specific conditions would section 60.6's "migrate the lowest-risk piece first" advice not apply?

---

## Answers

**1.** (a) Not architectural — trivial to reverse, affects nothing else. (b) Architectural — expensive to reverse (data residency, latency to every other system), affects everything deployed there. (c) Not architectural — costs nothing to change, affects one chart. (d) Architectural — every service on both ends has to agree on it, and changing it later means coordinating every consumer.

**2.** Context answers "what is this system for, and who does it talk to?" — the question a new employee or a board member asks. Container answers "what are its major moving pieces, and how do they connect?" — the question an engineer joining the team asks. A context diagram can't tell you which pipeline runs the ingestion; a container diagram crammed with every external person and system becomes unreadable.

**3.** (a) "The dashboard's initial view renders in under 2 seconds, p95 over a week, measured from the BI tool's own performance log." (b) "No credential can access data outside its assigned scope; access reviewed quarterly" (specific to whatever "secure" means for this system — always ask secure against what). (c) "Every write carries an idempotency key and is retried safely on failure, and every load is reconciled against the source daily; zero unexplained row-count differences, checked every night."

**4.** Title and number; status and date; context; decision; alternatives considered; consequences; owner and revisit trigger.

**5.** Without it, the same debate happens again months later when someone new joins and asks "why don't we just use X?" — and nobody remembers that X was already considered and rejected, so the organization re-spends the time re-litigating a closed question, sometimes reversing it based on incomplete information the first decision-makers already had and accounted for.

**6–10.** Personal exercises with no single correct answer; check against the chapter's criteria — a context diagram with all arrows crossing the one central box; a container diagram whose pieces are genuinely independently deployable; NFRs with numbers and named measurement methods; a trace that follows one real request without gaps; an ADR with real, specific rejected alternatives.

**11.** The new line reads `99.95%: 4.38 hours a year, 21.9 minutes a month, 5.1 minutes a week`. So 99.95% allows about 22 minutes of downtime a month, against 3.65 hours at 99.5%: ten times less. Appropriateness depends entirely on the system: a personal analytics dashboard rarely justifies 99.95% (the cost of achieving it — redundant infrastructure, and **on-call rotations**, people taking turns to be reachable out of hours — usually exceeds the cost of the rare outage); a payment system or a safety alert often does.

**12.** Benefit: a failure in one (say, the defect model's serving endpoint going down) can't cascade into the others, and each can be redeployed, scaled, or even retired on its own schedule without coordinating with the other two teams. Cost: duplicated infrastructure (two separate serving setups, as the story found), which is wasted spend and operational overhead once there are enough of them to matter — exactly the trade-off Meera revisited by consolidating serving, while deliberately keeping the containers' logical independence.

**13.** Ask: what specific, named problem does the current orchestrator actually have that the new one solves? What would migration cost — in time, in running two systems in parallel, in retraining the team? Is there a real incident or limitation driving this, or only that the new tool "looks better"? If there's no named problem, this is resume-driven design (section 60.6's mistake table), and the answer is no, not yet — revisit if a real limitation appears.

**14.** A level 3 diagram of the PO-intake pipeline would show the seven Chapter 58 stages (read, extract, parse, validate, decide, write, report) as internal components, with arrows showing which stage calls which and where the exception queue branches off. A level 4 diagram of "validate" would need to show the actual validation functions and their logic — the chapter recommends against hand-drawing this because it changes with nearly every code commit, so a manually maintained version goes stale within weeks and becomes actively misleading, worse than no diagram at all.

**15.** Following section 60.6: (1) run both serving setups side by side for a defined period, comparing outputs on the same inputs; (2) migrate the lower-risk model first — as Meera did with the defect model, whose failure the QC inspector could cover by hand for a shift, rather than the support assistant, whose wrong answers reach customers; (3) keep the old serving path live and able to take traffic until the new one has run through at least one full realistic cycle; (4) write the ADR recording the consolidation decision, what was rejected (keeping them separate, or migrating the riskier one first), and why.

**16.** For example, the "exception queue has no defined maximum wait time" risk: investigate by load-testing the confirmation step against a simulated festive-season email volume (using Chapter 58's measured review times), and set an explicit **SLA** (service-level agreement, Chapter 46, section 46.10: a stated promise of when something will be done) once you know the real relationship between volume and queue length. The risk can be removed from the list once that SLA is defined, tested under realistic load, and either met or has a documented mitigation (more reviewers, an escalation path) for when it isn't.

**17.** Explain, briefly, what the ADR protects against — not disagreement now, but the same conversation happening again in eight months with nobody remembering why. Offer to write a genuinely short version (a title, one-sentence decision, and a two-line "why," which takes five minutes) rather than insisting on the full seven-part template every time — a short ADR that exists beats a thorough one that doesn't get written because it seemed like too much overhead.

**18.** A big-bang rewrite can be the right call when the existing system is actively unsafe to keep running (a security vulnerability with no incremental patch path), when it's built on infrastructure that's being discontinued with a hard deadline, or when the team and budget genuinely support running two full systems in parallel for the migration period. The "migrate lowest-risk piece first" advice doesn't apply when there's no way to run part of the old system and part of the new one at once — some technology changes are genuinely all-or-nothing — but this is rarer than most rewrite proposals assume, and the burden of proof should be on showing that a piecewise migration is truly impossible, not merely inconvenient.

---

## Where this leads

- **Chapter 61, Distributed Systems & Trade-offs:** what happens when the container diagram's boxes are on different machines — consistency, availability, and the failures Part 5's stack has actually had.
- **Chapter 62, Data Architecture Patterns:** naming and comparing the shapes (Lambda, Kappa, medallion, data mesh) that a warehouse-and-pipeline design like section 60.5's actually follows.
- **Chapter 63, Automation Architecture & Governance:** the eight-container platform, inventoried, owned, and governed at company scale — building directly on the container diagram this chapter drew.
- **Chapter 64, Security, Privacy, Governance & Responsible AI:** closing the "access control model not yet defined" risk this chapter's design document left open.
- **Chapter 65, FinOps: The Economics of Data Platforms:** pricing the platform bottom-up, and testing the cost ceiling this chapter set top-down.
- **Chapter 67, The Architect as Leader:** influencing people you don't manage, worked through in section 67.4's influencing-without-authority scenario; Vikram's one question in this chapter's story, which stopped a rewrite he had no authority to veto, is the same move.
- **Interview preparation:** the Architecture & Leadership Question Bank (Chapter 80) asks for C4 diagrams and ADRs directly; Chapter 77's system-design questions ("walk me through how you'd design X") are answered with this chapter's NFR method.